# Лабораторная работа №4. Анализ систем синтеза речи

Ноутбук повторяет вычислительную часть отчёта: читает CSV с разметкой, считает задержки и проценты ошибок, а также проверяет наличие сохранённых WAV-файлов для пяти TTS из плана.

In [ ]:
from pathlib import Path
import csv

LAB_DIR = Path.cwd() if Path.cwd().name == 'lab4' else Path.cwd() / 'lab4'
DATA_DIR = LAB_DIR / 'data'
AUDIO_DIR = LAB_DIR / 'audio'
SYSTEMS = ['silero', 'rhvoice', 'piper', 'coqui_xtts', 'espeak_ng']
SYSTEM_NAMES = ['Silero TTS', 'RHVoice', 'Piper', 'Coqui XTTS', 'eSpeak NG']
LAB_DIR

In [ ]:
def read_csv(path):
    with path.open(encoding='utf-8', newline='') as file:
        return list(csv.DictReader(file))

latency_rows = read_csv(DATA_DIR / 'latency.csv')
results_rows = read_csv(DATA_DIR / 'results.csv')
manifest_rows = read_csv(AUDIO_DIR / 'manifest.csv')
len(latency_rows), len(results_rows), len(manifest_rows)

In [ ]:
from statistics import mean
from collections import defaultdict

latency = defaultdict(list)
for row in latency_rows:
    latency[row['system']].append(float(row['latency_seconds']))

for system in SYSTEM_NAMES:
    values = latency[system]
    print(f'{system:12s} mean={mean(values):.2f}s min={min(values):.2f}s max={max(values):.2f}s')

In [ ]:
quality = defaultdict(lambda: {'targets': 0, 'errors': 0})
for row in results_rows:
    key = (row['system'], row['category'])
    quality[key]['targets'] += int(row['target_count'])
    quality[key]['errors'] += int(row['error_count'])

for category in ['graph_abbrev', 'initial_abbrev', 'numbers']:
    print('\n' + category)
    for system in SYSTEM_NAMES:
        item = quality[(system, category)]
        percent = 100 * item['errors'] / item['targets']
        print(f'{system:12s} {item["errors"]:2d}/{item["targets"]:2d} = {percent:5.2f}%')

In [ ]:
for engine in SYSTEMS:
    files = sorted((AUDIO_DIR / engine).rglob('*.wav'))
    print(f'{engine:10s}: {len(files)} wav files')
    assert len(files) == 31, f'{engine}: expected 31 files, got {len(files)}'

assert len(manifest_rows) == 155
print('Audio manifest is complete.')